# Metric Summary

This notebook summarizes released automatic evaluation results for the XSAMSum subset.
It prefers committed metric files and otherwise falls back to aggregating the released annotation table.

In [2]:
from pathlib import Path
import pandas as pd

def find_root(start: Path) -> Path:
    for path in (start.resolve(), *start.resolve().parents):
        if (path / "outputs").exists() and (path / "annotations").exists():
            return path
    raise FileNotFoundError("Could not locate repository root.")

ROOT = find_root(Path.cwd())
ROOT


PosixPath('/Users/yunu919/Desktop/grounded-semantic-pipeline')

In [3]:
corpus_path = ROOT / "outputs" / "metrics" / "corpus_scores_zh_XSAMSum.csv"
agg_path = ROOT / "outputs" / "metrics" / "aggregated_error_rates.csv"
annotation_path = ROOT / "annotations" / "qualitative_direct.csv"

if corpus_path.exists():
    corpus_df = pd.read_csv(corpus_path)
else:
    ann = pd.read_csv(annotation_path)
    corpus_df = (
        ann.groupby("model_name", as_index=False)[["rouge1", "rouge2", "rougeL", "bs_f1_raw"]]
        .mean()
        .round(2)
    )

agg_df = pd.read_csv(agg_path)
corpus_df


,model_name,rouge1,rouge2,rougeL,bs_f1_raw
0,aya-expanse:8b,26.53,7.89,20.84,70.11
1,gemma4:e4b,30.27,8.32,23.68,70.85
2,mBART-large,41.95,18.10,34.85,77.64
3,qwen3.5:9b,28.35,7.62,22.42,70.47


In [4]:
agg_df


,model_name,no_error,hallucination,omission,language
0,aya-expanse:8b,0.38,0.43,0.07,0.17
1,gemma4:e4b,0.63,0.29,0.08,0.00
2,qwen3.5:9b,0.58,0.39,0.03,0.00
3,mBART-large,0.20,0.61,0.48,0.11


In [5]:
corpus_df.merge(agg_df, on="model_name", how="left").sort_values("rougeL", ascending=False)


,model_name,rouge1,rouge2,rougeL,bs_f1_raw,no_error,hallucination,omission,language
2,mBART-large,41.95,18.10,34.85,77.64,0.20,0.61,0.48,0.11
1,gemma4:e4b,30.27,8.32,23.68,70.85,0.63,0.29,0.08,0.00
3,qwen3.5:9b,28.35,7.62,22.42,70.47,0.58,0.39,0.03,0.00
0,aya-expanse:8b,26.53,7.89,20.84,70.11,0.38,0.43,0.07,0.17
